# 00 — GBIF Occurrence Data

Download and filter Swiss plant occurrences from the GBIF open data snapshot on S3.

In [1]:
import pyarrow.parquet as pq
import s3fs

s3 = s3fs.S3FileSystem(anon=True)

example_file = "s3://gbif-open-data-eu-central-1/occurrence/2024-03-01/occurrence.parquet/000001"

with s3.open(example_file) as f:
    schema = pq.read_schema(f)
    column_names = schema.names

print(f"The dataset has {len(column_names)} columns.")
print("Columns:", column_names)

The dataset has 50 columns.
Columns: ['gbifid', 'datasetkey', 'occurrenceid', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'infraspecificepithet', 'taxonrank', 'scientificname', 'verbatimscientificname', 'verbatimscientificnameauthorship', 'countrycode', 'locality', 'stateprovince', 'occurrencestatus', 'individualcount', 'publishingorgkey', 'decimallatitude', 'decimallongitude', 'coordinateuncertaintyinmeters', 'coordinateprecision', 'elevation', 'elevationaccuracy', 'depth', 'depthaccuracy', 'eventdate', 'day', 'month', 'year', 'taxonkey', 'specieskey', 'basisofrecord', 'institutioncode', 'collectioncode', 'catalognumber', 'recordnumber', 'identifiedby', 'dateidentified', 'license', 'rightsholder', 'recordedby', 'typestatus', 'establishmentmeans', 'lastinterpreted', 'mediatype', 'issue']


In [2]:
import config
import duckdb
from geo_utils import BOUNDS

S3_BUCKET_PATH = "s3://gbif-open-data-eu-central-1/occurrence/2026-04-01/occurrence.parquet/*"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute("SET s3_region='eu-central-1';")

# Write to S3 via local temp file, then upload
local_path = '/tmp/swiss_occurences.parquet'

query = f"""
COPY (
  SELECT
    gbifid, species, specieskey, decimallatitude, decimallongitude,
    coordinateuncertaintyinmeters, eventdate, occurrencestatus
  FROM read_parquet('{S3_BUCKET_PATH}')
  WHERE occurrencestatus = 'PRESENT'
    AND decimallatitude IS NOT NULL
    AND decimallongitude IS NOT NULL
    AND kingdom = 'Plantae'
    AND decimallatitude  BETWEEN {BOUNDS['lat_min']} AND {BOUNDS['lat_max']}
    AND decimallongitude BETWEEN {BOUNDS['lon_min']} AND {BOUNDS['lon_max']}
) TO '{local_path}' (FORMAT PARQUET);
"""

print(f"Filtering to extent: {BOUNDS}")
con.execute(query)

# Upload to S3
import s3fs
s3 = s3fs.S3FileSystem(anon=False)
s3.put(local_path, config.GBIF_PARQUET)
print(f"Saved to {config.GBIF_PARQUET}")


Filtering to extent: {'lat_min': 45.56927314539999, 'lat_max': 48.07159314539999, 'lon_min': 5.7157257961, 'lon_max': 10.6627057961}
Saved to s3://km-cas-datalake/processed/gbif/swiss_occurences.parquet
